# Boosting Project:

In [ ]:
import pandas as pd
import numpy as np
import codecademylib3

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

path_to_data = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

col_names = [
    'age', 'workclass', 'fnlwgt','education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain','capital-loss',
    'hours-per-week','native-country', 'income'
]

df = pd.read_csv(path_to_data, header=None, names = col_names)
#print(df.head())

#Clean columns by stripping extra whitespace for columns of type "object"
for c in df.select_dtypes(include=['object']).columns:
    df[c] = df[c].str.strip()

target_column = "income"
raw_feature_cols = [
    'age',
    'education-num',
    'workclass',
    'hours-per-week',
    'sex',
    'race'
]

##1. Percentage of samples with income < and > 50k
target_col = df[target_column].value_counts(normalize=True)
#print(target_col)

##2. Data types of features
dtypes = df[raw_feature_cols].dtypes
#print(dtypes)

##3. Preparing the features
X = pd.get_dummies(df[raw_feature_cols], drop_first=True)
X.head(n=5)


##4. Convert target variable to binary
y = (df['income'].apply(lambda x: 0 if x == '<=50K' else 1))

##5a. Create train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, test_size=.3)

##5b. Create base estimator and store it as decision_stump
decision_stump = DecisionTreeClassifier(max_depth=1)

##6. Create AdaBoost Classifier
ada_classifier = AdaBoostClassifier(base_estimator=decision_stump)

##7. Create GradientBoost Classifier
grad_classifier = GradientBoostingClassifier()

##8a.Fit models and get predictions
ada_classifier.fit(X_train, y_train)
grad_classifier.fit(X_train, y_train)

y_pred_ada = ada_classifier.predict(X_test)
y_pred_grad = grad_classifier.predict(X_test)

ada_accuracy = accuracy_score(y_test, y_pred_ada)
grad_accuracy = accuracy_score(y_test, y_pred_ada)

ada_f1 = f1_score(y_test, y_pred_ada)
grad_f1 = f1_score(y_test, y_pred_grad)

##8b. Print accuracy and F1
print(f"AdaBoosting - Accuracy Score: {ada_accuracy.round(4)}, F1 Score: {ada_f1.round(4)}")

print(f"Gradient Boosting - Accuracy Score: {grad_accuracy.round(4)}, F1 Score: {grad_f1.round(4)}")

##9. Hyperparameter Tuning
n_estimators_list = [10, 30, 50, 70, 90]
from sklearn.model_selection import GridSearchCV

ada_cv = GridSearchCV(ada_classifier,{'n_estimators':n_estimators_list})

ada_cv.fit(X_train, y_train)

#ada_scores_list 
ada_scores_list = ada_cv.cv_results_['mean_test_score']
# print(ada_scores_list)

##10. Plot mean test scores
plt.plot(n_estimators_list, ada_scores_list, marker='o')
plt.xlabel('n_estimators')
plt.ylabel('mean_test_score')
plt.title('AdaBoost Performance vs n_estimators')
plt.grid(True)
plt.show()